# Imputation Technique 4: Mean / Median Imputation

**Dataset:** `Loan_Default.csv`

**When to use:** For **numerical** features. Assumes data is **Missing At Random (MCAR)**.

**Key concept:**

- **Mean:** Replace NaN with the arithmetic average. Best for **normally distributed** data.
- **Median:** Replace NaN with the middle value. Best for **skewed** distributions or when **outliers** are present.

---


### Step 1: Setup — Data Loading & Prep


In [2]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

df_prep = df.copy()
for c in Nominal_features:
    df_prep[c + '_freq'] = df_prep[c].map(df_prep.groupby(c).size() / df_prep.shape[0])
    indexer = pd.factorize(df_prep[c], sort=True)[1]
    df_prep[c] = indexer.get_indexer(df_prep[c])
df_prep = df_prep.drop(Nominal_features, axis=1)

high_missing_cols = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'property_value', 'LTV', 'dtir1']
df_drop = df_prep.drop(high_missing_cols, axis=1)

print(f'Starting shape: {df_drop.shape}')

Starting shape: (148670, 26)


### Step 2: Define Columns to Impute


In [3]:
Cols_to_be_imputed = [
    'term', 'income', 'age',
    'loan_limit_freq', 'approv_in_adv_freq', 'loan_purpose_freq',
    'Neg_ammortization_freq', 'submission_of_application_freq'
]

### Step 3: Apply Mean Imputation

Using a manual loop with `fillna(col.mean())`.


In [4]:
df_mean = df_drop.copy()

for c in Cols_to_be_imputed:
    df_mean[c].fillna(df_mean[c].mean(), inplace=True)

df_mean.head()

/tmp/ipykernel_11286/3297564543.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mean[c].fillna(df_mean[c].mean(), inplace=True)


,loan_amount,term,income,Credit_Score,age,Status,loan_limit_freq,Gender_freq,approv_in_adv_freq,loan_type_freq,...,lump_sum_payment_freq,construction_type_freq,occupancy_type_freq,Secured_by_freq,total_units_freq,credit_type_freq,co-applicant_credit_type_freq,submission_of_application_freq,Region_freq,Security_Type_freq
0,116500,360.0,1740.0,758,0.0,1,0.910392,0.253306,0.838239,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.644474,0.430591,0.999778
1,206500,360.0,4980.0,552,3.0,1,0.910392,0.284832,0.838239,0.139652,...,0.022762,0.999778,0.929582,0.999778,0.985269,0.102899,0.499617,0.644474,0.502603,0.999778
2,406500,360.0,9480.0,834,1.0,0,0.910392,0.284832,0.155653,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.644474,0.430591,0.999778
3,456500,360.0,11880.0,587,2.0,0,0.910392,0.284832,0.838239,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.354180,0.502603,0.999778
4,696500,360.0,10440.0,602,0.0,0,0.910392,0.278462,0.155653,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.295292,0.499617,0.354180,0.502603,0.999778


### Step 4: Apply Median Imputation (Alternative)

Better for skewed data like `income`.


In [5]:
df_median = df_drop.copy()

for c in Cols_to_be_imputed:
    df_median[c].fillna(df_median[c].median(), inplace=True)

print(f"Mean of income:   {df_drop['income'].mean():.2f}")
print(f"Median of income: {df_drop['income'].median():.2f}")

Mean of income:   6957.34
Median of income: 5760.00


/tmp/ipykernel_11286/2802753568.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_median[c].fillna(df_median[c].median(), inplace=True)
/tmp/ipykernel_11286/2802753568.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

### Step 5: Verify Results


In [6]:
missing_after = df_mean.isna().sum()
print('Missing values after Mean Imputation:')
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else 'None — all filled!')

Missing values after Mean Imputation:
None — all filled!
